In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
data=pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')

In [ ]:
data = np.array(data)
m,n=data.shape
np.random.shuffle(data)

data_test= data[0:1000].T
Y_test= data_test[0]
X_test= data_test[1:n]
X_test=X_test/255

data_train=data[1000:m].T
Y_train=data_train[0]
X_train=data_train[1:n]
X_train=X_train/255
_,m_train = X_train.shape


In [ ]:
Y_train

array([5, 4, 8, ..., 9, 4, 7], shape=(41000,))

In [ ]:
def initialize():
    W1= np.random.rand(10,784)-0.5
    b1= np.random.rand(10,1)-0.5
    W2= np.random.rand(10,10)-0.5
    b2= np.random.rand(10,1)-0.5
    return W1,b1,W2,b2
def ReLu(z):
    return np.maximum(z,0)
def softmax(z):
    A=np.exp(z)/sum(np.exp(z))
    return A
def forward_prop(W1,b1,W2,b2,X):
    Z1= W1.dot(X)+b1
    A1=ReLu(Z1)
    Z2=W2.dot(A1)+b2
    A2=softmax(Z2)
    return Z1,A1,Z2,A2
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y
def deriv(z):
    return z>0
def back_prop(Z1,A1,Z2,A2,W1,W2,X,Y):
    one_hot_Y=one_hot(Y)
    m=Y.size
    dZ2=A2-one_hot_Y
    dW2=(1/m)*dZ2.dot(A1.T)
    db2=(1/m)* np.sum(dZ2)
    dZ1=W2.T.dot(dZ2)*deriv(Z1)
    dW1= (1/m)*dZ1.dot(X.T)
    db1= (1/m)* np.sum(dZ1)
    return dW1,db1,dW2,db2
def update(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha=0.1):
    W1=W1-alpha*dW1
    W2=W2-alpha*dW2
    b1=b1-alpha*db1
    b2=b2-alpha*db2
    return W1,b1,W2,b2
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size
def gradient_descent(X, Y, alpha=0.1, iterations=1000):
    W1,b1,W2,b2= initialize()
    for i in range(iterations):
        Z1,A1,Z2,A2=forward_prop(W1,b1,W2,b2,X)
        dW1,db1,dW2,db2=back_prop(Z1,A1,Z2,A2,W1,W2,X,Y)
        W1,b1,W2,b2=update(W1,b1,W2,b2,dW1,db1,dW2,db2)
        if i%10==0:
            print("Iteration: ",i)
            print(get_accuracy(get_predictions(A2),Y))
    return W1,b1,W2,b2
    
    

In [ ]:
W1,b1,W2,b2=gradient_descent(X_train,Y_train)

Iteration:  0
[4 6 6 ... 6 6 4] [5 4 8 ... 9 4 7]
0.09539024390243903
Iteration:  10
[4 6 6 ... 6 6 0] [5 4 8 ... 9 4 7]
0.20336585365853657
Iteration:  20
[4 6 6 ... 6 6 0] [5 4 8 ... 9 4 7]
0.32934146341463416
Iteration:  30
[4 6 1 ... 6 6 0] [5 4 8 ... 9 4 7]
0.4256341463414634
Iteration:  40
[4 6 1 ... 6 4 7] [5 4 8 ... 9 4 7]
0.49053658536585365
Iteration:  50
[7 6 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.5381707317073171
Iteration:  60
[7 6 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.5745853658536585
Iteration:  70
[7 6 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.6069024390243902
Iteration:  80
[7 6 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.632219512195122
Iteration:  90
[7 6 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.6529024390243903
Iteration:  100
[7 6 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.6710975609756098
Iteration:  110
[7 9 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.6867560975609757
Iteration:  120
[7 9 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.7008048780487804
Iteration:  130
[7 9 1 ... 9 4 7] [5 4 8 ... 9 4 7]
0.7134146341463414
Iteration:  14

In [ ]:
def make_predictions(X,W1,b1,W2,b2):
    _,_,_,A2 = forward_prop(W1,b1,W2,b2,X)
    predictions=get_predictions(A2)
    return predictions
test= make_predictions(X_train,W1,b1,W2,b2) 
print(get_accuracy(test,Y_train))


[5 9 8 ... 9 4 9] [5 4 8 ... 9 4 7]
0.886219512195122


In [ ]:
def categorical_cross_entropy_loss(A2,Y):
    m=Y.size
    one_hot_Y=one_hot(Y)
    A2_clipped=np.clip(A2,1e-15,1-1e-15) #to avoid log(0) error 
    loss=-np.sum(one_hot_Y*np.log(A2_clipped))/m
    return loss
    
def gradient_with_adam(X,Y,alpha=0.001,iterations=1000):
    W1,b1,W2,b2=initialize()
    m=Y.size
    #hyperparameters
    beta1=0.9
    beta2=0.999
    epsilon=1e-10
    m_dW1, m_db1 = np.zeros_like(W1), np.zeros_like(b1)
    m_dW2, m_db2 = np.zeros_like(W2), np.zeros_like(b2)
    v_dW1, v_db1 = np.zeros_like(W1), np.zeros_like(b1)
    v_dW2, v_db2 = np.zeros_like(W2), np.zeros_like(b2)
    t=0
    for i in range(iterations):
        Z1,A1,Z2,A2=forward_prop(W1,b1,W2,b2,X)
        dW1,db1,dW2,db2=back_prop(Z1,A1,Z2,A2,W1,W2,X,Y)
        t+=1 #bias correction
        #layer 1
        m_dW1 = beta1*m_dW1+(1-beta1)*dW1
        m_db1 = beta1*m_db1+(1-beta1)*db1
        v_dW1 = beta2*v_dW1+(1-beta2)*(dW1**2)
        v_db1 = beta2*v_db1+(1-beta2)*(db1** 2)
        m_dW1_corr=m_dW1/(1-beta1**t)
        m_db1_corr=m_db1/(1-beta1**t)
        v_dW1_corr=v_dW1/(1-beta2**t)
        v_db1_corr=v_db1/(1-beta2**t)
        W1-=alpha*m_dW1_corr/(np.sqrt(v_dW1_corr)+epsilon)
        b1-=alpha*m_db1_corr/(np.sqrt(v_db1_corr)+epsilon)
        #layer 2
        m_dW2=beta1*m_dW2+(1-beta1)*dW2
        m_db2=beta1*m_db2+(1-beta1)*db2
        v_dW2=beta2*v_dW2+(1-beta2)*(dW2**2)
        v_db2=beta2*v_db2+(1-beta2)*(db2**2)
        m_dW2_corr=m_dW2/(1-beta1**t)
        m_db2_corr=m_db2/(1-beta1**t)
        v_dW2_corr=v_dW2/(1-beta2**t)
        v_db2_corr=v_db2/(1-beta2**t)
        W2-=alpha*m_dW2_corr/(np.sqrt(v_dW2_corr)+epsilon)
        b2-=alpha*m_db2_corr/(np.sqrt(v_db2_corr)+epsilon)


        if i%10==0:
            print("iteration",i)
            loss= categorical_cross_entropy_loss(A2,Y)
            accuracy=get_accuracy(get_predictions(A2),Y)
            print("Accuracy= ",accuracy, "  ","categorical_cross_entropy_loss= ",loss)
    return W1,b1,W2,b2
            


In [ ]:
W1,b1,W2,b2=gradient_with_adam(X_train,Y_train)

iteration 0
[0 0 3 ... 6 0 0] [3 8 5 ... 3 1 0]
Accuracy=  0.10995121951219512    categorical_cross_entropy_loss=  4.129255511927536
iteration 10
[5 0 3 ... 6 0 0] [3 8 5 ... 3 1 0]
Accuracy=  0.12324390243902439    categorical_cross_entropy_loss=  3.23807580202005
iteration 20
[5 2 3 ... 6 0 0] [3 8 5 ... 3 1 0]
Accuracy=  0.13914634146341465    categorical_cross_entropy_loss=  2.7709747076959834
iteration 30
[5 2 3 ... 6 2 0] [3 8 5 ... 3 1 0]
Accuracy=  0.17373170731707316    categorical_cross_entropy_loss=  2.4850151952917456
iteration 40
[0 2 3 ... 6 2 0] [3 8 5 ... 3 1 0]
Accuracy=  0.20968292682926828    categorical_cross_entropy_loss=  2.2994739303454477
iteration 50
[0 2 3 ... 5 2 0] [3 8 5 ... 3 1 0]
Accuracy=  0.23073170731707318    categorical_cross_entropy_loss=  2.174475198645005
iteration 60
[0 2 7 ... 5 2 0] [3 8 5 ... 3 1 0]
Accuracy=  0.2524390243902439    categorical_cross_entropy_loss=  2.0807776026363056
iteration 70
[0 2 7 ... 7 2 0] [3 8 5 ... 3 1 0]
Accuracy=  0

In [ ]:

_,_,_,A2_test=forward_prop(W1,b1,W2,b2,X_test)
test_accuracy=get_accuracy(get_predictions(A2_test),Y_test)

print(f"Final Training Accuracy: 90.6%")
print(f"Final Test Accuracy:     {test_accuracy * 100:.2f}%")

[3 0 6 3 7 2 7 8 2 6 0 0 9 6 6 6 0 5 5 2 7 3 9 9 6 5 4 9 1 4 4 9 9 8 5 3 7
 3 8 1 8 9 9 6 8 9 8 9 2 8 9 1 0 2 4 7 5 6 8 6 3 5 5 6 6 4 0 6 6 7 1 6 1 4
 0 1 4 4 4 4 8 0 5 8 4 0 7 3 7 3 0 1 3 1 6 4 2 9 5 8 8 3 4 5 4 7 3 1 3 0 2
 7 4 0 2 1 7 9 9 2 3 0 3 1 2 7 2 8 5 1 2 7 8 7 3 7 0 7 0 3 5 6 0 0 6 7 9 0
 0 5 4 1 0 1 8 4 7 4 4 5 9 3 3 3 7 7 5 8 0 8 7 7 3 6 2 0 6 8 3 5 3 6 9 6 0
 3 7 5 8 4 6 6 8 2 2 2 4 8 2 3 9 8 8 4 9 4 2 1 9 5 6 5 6 6 0 4 8 2 7 6 2 7
 4 5 6 0 2 7 0 7 4 2 7 3 3 8 8 1 7 6 5 5 4 2 0 1 6 8 2 8 6 0 9 9 6 5 1 0 6
 7 3 2 1 6 5 3 4 3 8 6 7 6 8 4 1 6 2 4 8 3 1 3 1 5 1 1 2 4 9 1 1 4 6 0 6 1
 7 1 6 3 4 5 7 6 0 3 8 6 6 2 9 6 4 2 3 8 9 2 7 5 2 0 7 4 5 3 2 3 9 6 8 8 6
 5 0 8 2 7 7 2 5 6 2 6 0 5 2 3 0 2 9 1 7 0 0 5 5 9 9 4 7 9 7 2 0 9 9 0 8 4
 3 9 5 6 3 9 2 4 3 8 3 0 2 7 6 7 9 8 7 6 1 0 3 9 2 4 4 7 1 0 5 7 7 2 7 8 6
 3 7 6 3 4 0 8 4 0 5 8 0 0 9 4 3 4 4 4 6 7 0 4 0 2 0 5 6 1 8 2 9 3 5 1 0 2
 3 7 5 7 2 3 7 6 5 1 9 0 5 8 3 0 1 4 7 5 5 1 4 4 0 4 2 5 2 8 0 2 3 5 1 4 3
 6 6 3 5 2 6 2 0 3 4 0 9 

In [ ]:
def initialize():
    W1= np.random.randn(128,784)*np.sqrt(2.0/784)
    b1= np.random.randn(128,1)
    
    W2= np.random.randn(64,128)*np.sqrt(2.0/128)
    b2= np.random.randn(64,1)

    W3=np.random.randn(10,64)*np.sqrt(2.0/64)
    b3=np.zeros((10,1))
    
    return W1,b1,W2,b2,W3,b3
def ReLu(z):
    return np.maximum(z,0)
def softmax(z):
    A=np.exp(z)/sum(np.exp(z))
    return A
def forward_prop(W1,b1,W2,b2,W3,b3,X):
    Z1= W1.dot(X)+b1
    A1=ReLu(Z1)
    Z2=W2.dot(A1)+b2
    A2=ReLu(Z2)
    Z3=W3.dot(A2)+b3
    A3=softmax(Z3)
    return Z1,A1,Z2,A2,Z3,A3
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y
def deriv(z):
    return z>0
def back_prop(Z1,A1,Z2,A2,Z3,A3,W1,W2,W3,X,Y):
    one_hot_Y=one_hot(Y)
    m=Y.size
    dZ3=A3-one_hot_Y
    dW3=(1/m)*dZ3.dot(A2.T)
    db3=(1/m)*np.sum(dZ3,axis=1,keepdims=True)
    dZ2=W3.T.dot(dZ3)*deriv(Z2)
    dW2=(1/m)*dZ2.dot(A1.T)
    db2=(1/m)* np.sum(dZ2)
    dZ1=W2.T.dot(dZ2)*deriv(Z1)
    dW1= (1/m)*dZ1.dot(X.T)
    db1= (1/m)* np.sum(dZ1)
    return dW1,db1,dW2,db2,dW3,db3

def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size
def categorical_cross_entropy_loss(A2,Y):
    m=Y.size
    one_hot_Y=one_hot(Y)
    A2_clipped=np.clip(A2,1e-15,1-1e-15) #to avoid log(0) error 
    loss=-np.sum(one_hot_Y*np.log(A2_clipped))/m
    return loss
    
def gradient_with_adam_3(X,Y,alpha=0.001,iterations=500):
    W1,b1,W2,b2,W3,b3=initialize()
    m=Y.size
    #hyperparameters
    beta1=0.9
    beta2=0.999
    epsilon=1e-10
    m_dW1, m_db1 = np.zeros_like(W1), np.zeros_like(b1)
    m_dW2, m_db2 = np.zeros_like(W2), np.zeros_like(b2)
    m_dW3, m_db3 = np.zeros_like(W3), np.zeros_like(b3)
    v_dW1, v_db1 = np.zeros_like(W1), np.zeros_like(b1)
    v_dW2, v_db2 = np.zeros_like(W2), np.zeros_like(b2)
    v_dW3, v_db3 = np.zeros_like(W3), np.zeros_like(b3)

    t=0
    for i in range(iterations):
        Z1,A1,Z2,A2,Z3,A3=forward_prop(W1,b1,W2,b2,W3,b3,X)
        dW1,db1,dW2,db2,dW3,db3=back_prop(Z1,A1,Z2,A2,Z3,A3,W1,W2,W3,X,Y)
        t+=1 #bias correction
        #layer 1
        m_dW1 = beta1*m_dW1+(1-beta1)*dW1
        m_db1 = beta1*m_db1+(1-beta1)*db1
        v_dW1 = beta2*v_dW1+(1-beta2)*(dW1**2)
        v_db1 = beta2*v_db1+(1-beta2)*(db1** 2)
        m_dW1_corr=m_dW1/(1-beta1**t)
        m_db1_corr=m_db1/(1-beta1**t)
        v_dW1_corr=v_dW1/(1-beta2**t)
        v_db1_corr=v_db1/(1-beta2**t)
        W1-=alpha*m_dW1_corr/(np.sqrt(v_dW1_corr)+epsilon)
        b1-=alpha*m_db1_corr/(np.sqrt(v_db1_corr)+epsilon)
        #layer 2
        m_dW2=beta1*m_dW2+(1-beta1)*dW2
        m_db2=beta1*m_db2+(1-beta1)*db2
        v_dW2=beta2*v_dW2+(1-beta2)*(dW2**2)
        v_db2=beta2*v_db2+(1-beta2)*(db2**2)
        m_dW2_corr=m_dW2/(1-beta1**t)
        m_db2_corr=m_db2/(1-beta1**t)
        v_dW2_corr=v_dW2/(1-beta2**t)
        v_db2_corr=v_db2/(1-beta2**t)
        W2-=alpha*m_dW2_corr/(np.sqrt(v_dW2_corr)+epsilon)
        b2-=alpha*m_db2_corr/(np.sqrt(v_db2_corr)+epsilon)
        #layer 3
        m_dW3=beta1*m_dW3+(1-beta1)*dW3
        m_db3=beta1*m_db3+(1-beta1)*db3
        v_dW3=beta2*v_dW3+(1-beta2)*(dW3**2)
        v_db3=beta2*v_db3+(1-beta2)*(db3**2)
        m_dW3_corr=m_dW3/(1-beta1**t)
        m_db3_corr=m_db3/(1-beta1**t)
        v_dW3_corr=v_dW3/(1-beta2**t)
        v_db3_corr=v_db3/(1-beta2**t)
        W3-=alpha*m_dW3_corr/(np.sqrt(v_dW3_corr)+epsilon)
        b3-=alpha*m_db3_corr/(np.sqrt(v_db3_corr)+epsilon)


        if i%50==0:
            print("iteration",i)
            loss= categorical_cross_entropy_loss(A3,Y)
            accuracy=get_accuracy(get_predictions(A3),Y)
            print("Accuracy= ",accuracy, "  ","categorical_cross_entropy_loss= ",loss)
    return W1,b1,W2,b2,W3,b3
            

    
    

In [ ]:
W1,b1,W2,b2,W3,b3=gradient_with_adam_3(X_train,Y_train)

iteration 0
[9 9 0 ... 9 0 9] [3 8 5 ... 3 1 0]
Accuracy=  0.088    categorical_cross_entropy_loss=  2.8256659177556407
iteration 50
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.8964878048780488    categorical_cross_entropy_loss=  0.3641050661333255
iteration 100
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9365853658536586    categorical_cross_entropy_loss=  0.22245381890773125
iteration 150
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.955780487804878    categorical_cross_entropy_loss=  0.15711695229861378
iteration 200
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9679756097560975    categorical_cross_entropy_loss=  0.1132844229712125
iteration 250
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9765121951219512    categorical_cross_entropy_loss=  0.08378102400339375
iteration 300
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9829024390243902    categorical_cross_entropy_loss=  0.06269490685838217
iteration 350
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.98807

In [ ]:

_,_,_,_,_,A3_test=forward_prop(W1,b1,W2,b2,W3,b3,X_test)
test_accuracy=get_accuracy(get_predictions(A3_test),Y_test)

print(f"Final Training Accuracy: 99.57%")
print(f"Final Test Accuracy:     {test_accuracy * 100:.2f}%")

[3 0 6 5 7 2 7 8 2 6 5 5 9 6 6 6 0 5 9 3 7 3 9 9 6 5 4 7 1 4 4 4 9 8 5 3 7
 3 8 1 8 9 9 6 8 4 9 4 2 8 9 1 0 2 4 7 0 6 8 6 3 5 5 6 6 4 0 6 6 7 1 6 1 4
 0 1 2 4 4 4 8 0 5 8 4 0 7 3 9 3 0 1 3 1 6 4 2 9 5 8 8 3 4 5 4 7 3 1 3 0 3
 7 4 0 2 1 7 9 9 2 3 0 3 1 2 7 2 8 5 1 2 7 8 7 3 7 0 3 0 3 5 5 0 0 6 7 9 0
 0 5 4 1 0 1 8 4 7 4 4 5 9 3 3 9 7 7 5 8 0 8 7 7 3 6 2 0 6 8 8 5 3 6 7 6 0
 3 7 5 8 4 6 6 8 2 2 2 7 8 3 3 9 8 9 4 9 4 2 1 9 5 6 5 8 6 0 4 8 2 7 6 2 7
 4 5 6 0 2 7 0 7 4 2 7 3 3 8 8 1 7 6 5 5 4 2 0 1 6 8 2 8 6 0 9 7 6 5 1 0 6
 7 3 2 1 6 5 3 4 3 8 6 7 6 3 4 1 6 7 4 8 3 1 3 1 5 1 1 2 4 9 1 1 4 6 0 6 1
 7 1 6 3 4 5 7 6 0 3 8 6 6 2 7 6 4 2 3 8 9 2 7 5 2 0 7 4 5 3 2 3 9 6 8 8 6
 5 0 8 2 7 7 2 5 6 2 6 0 5 3 3 0 2 9 1 7 0 3 3 5 9 9 4 7 9 7 1 0 9 9 0 8 4
 3 9 9 6 3 9 2 5 3 8 3 0 2 7 2 7 7 8 7 6 1 0 3 9 2 4 4 7 1 0 5 7 7 6 7 5 6
 3 7 6 3 4 0 2 4 0 5 8 0 0 9 4 3 4 4 4 6 7 0 4 0 2 0 5 6 1 8 2 9 3 5 1 0 2
 3 7 5 7 2 3 7 6 8 1 9 0 6 8 3 0 1 4 7 3 3 1 4 4 0 4 2 5 2 8 0 2 3 5 1 4 8
 6 6 3 5 3 6 2 0 3 4 0 9 

In [ ]:
def initialize():
    W1= np.random.randn(128,784)*np.sqrt(2.0/784)
    b1= np.random.randn(128,1)
    
    W2= np.random.randn(64,128)*np.sqrt(2.0/128)
    b2= np.random.randn(64,1)

    W3=np.random.randn(10,64)*np.sqrt(2.0/64)
    b3=np.zeros((10,1))
    
    return W1,b1,W2,b2,W3,b3
def ReLu(z):
    return np.maximum(z,0)
def softmax(z):
    A=np.exp(z)/sum(np.exp(z))
    return A
def forward_prop(W1,b1,W2,b2,W3,b3,X):
    Z1= W1.dot(X)+b1
    A1=ReLu(Z1)
    Z2=W2.dot(A1)+b2
    A2=ReLu(Z2)
    Z3=W3.dot(A2)+b3
    A3=softmax(Z3)
    return Z1,A1,Z2,A2,Z3,A3
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y
def deriv(z):
    return z>0
def back_prop(Z1,A1,Z2,A2,Z3,A3,W1,W2,W3,X,Y): # with overfitting L2 regularization penalty 
    one_hot_Y=one_hot(Y)
    m=Y.size
    dZ3=A3-one_hot_Y
    dW3=(1/m)*dZ3.dot(A2.T)*(0.01/m)*W3
    db3=(1/m)*np.sum(dZ3,axis=1,keepdims=True)
    dZ2=W3.T.dot(dZ3)*deriv(Z2)
    dW2=(1/m)*dZ2.dot(A1.T)*(0.005/m)*W2
    db2=(1/m)* np.sum(dZ2)
    dZ1=W2.T.dot(dZ2)*deriv(Z1)
    dW1= (1/m)*dZ1.dot(X.T)
    db1= (1/m)* np.sum(dZ1)
    return dW1,db1,dW2,db2,dW3,db3

def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size
def categorical_cross_entropy_loss(A2,Y):
    m=Y.size
    one_hot_Y=one_hot(Y)
    A2_clipped=np.clip(A2,1e-15,1-1e-15) #to avoid log(0) error 
    loss=-np.sum(one_hot_Y*np.log(A2_clipped))/m
    return loss
    
def gradient_with_adam_3(X,Y,alpha=0.001,iterations=500):
    W1,b1,W2,b2,W3,b3=initialize()
    m=Y.size
    #hyperparameters
    beta1=0.9
    beta2=0.999
    epsilon=1e-10
    m_dW1, m_db1 = np.zeros_like(W1), np.zeros_like(b1)
    m_dW2, m_db2 = np.zeros_like(W2), np.zeros_like(b2)
    m_dW3, m_db3 = np.zeros_like(W3), np.zeros_like(b3)
    v_dW1, v_db1 = np.zeros_like(W1), np.zeros_like(b1)
    v_dW2, v_db2 = np.zeros_like(W2), np.zeros_like(b2)
    v_dW3, v_db3 = np.zeros_like(W3), np.zeros_like(b3)

    t=0
    for i in range(iterations):
        Z1,A1,Z2,A2,Z3,A3=forward_prop(W1,b1,W2,b2,W3,b3,X)
        dW1,db1,dW2,db2,dW3,db3=back_prop(Z1,A1,Z2,A2,Z3,A3,W1,W2,W3,X,Y)
        t+=1 #bias correction
        #layer 1
        m_dW1 = beta1*m_dW1+(1-beta1)*dW1
        m_db1 = beta1*m_db1+(1-beta1)*db1
        v_dW1 = beta2*v_dW1+(1-beta2)*(dW1**2)
        v_db1 = beta2*v_db1+(1-beta2)*(db1** 2)
        m_dW1_corr=m_dW1/(1-beta1**t)
        m_db1_corr=m_db1/(1-beta1**t)
        v_dW1_corr=v_dW1/(1-beta2**t)
        v_db1_corr=v_db1/(1-beta2**t)
        W1-=alpha*m_dW1_corr/(np.sqrt(v_dW1_corr)+epsilon)
        b1-=alpha*m_db1_corr/(np.sqrt(v_db1_corr)+epsilon)
        #layer 2
        m_dW2=beta1*m_dW2+(1-beta1)*dW2
        m_db2=beta1*m_db2+(1-beta1)*db2
        v_dW2=beta2*v_dW2+(1-beta2)*(dW2**2)
        v_db2=beta2*v_db2+(1-beta2)*(db2**2)
        m_dW2_corr=m_dW2/(1-beta1**t)
        m_db2_corr=m_db2/(1-beta1**t)
        v_dW2_corr=v_dW2/(1-beta2**t)
        v_db2_corr=v_db2/(1-beta2**t)
        W2-=alpha*m_dW2_corr/(np.sqrt(v_dW2_corr)+epsilon)
        b2-=alpha*m_db2_corr/(np.sqrt(v_db2_corr)+epsilon)
        #layer 3
        m_dW3=beta1*m_dW3+(1-beta1)*dW3
        m_db3=beta1*m_db3+(1-beta1)*db3
        v_dW3=beta2*v_dW3+(1-beta2)*(dW3**2)
        v_db3=beta2*v_db3+(1-beta2)*(db3**2)
        m_dW3_corr=m_dW3/(1-beta1**t)
        m_db3_corr=m_db3/(1-beta1**t)
        v_dW3_corr=v_dW3/(1-beta2**t)
        v_db3_corr=v_db3/(1-beta2**t)
        W3-=alpha*m_dW3_corr/(np.sqrt(v_dW3_corr)+epsilon)
        b3-=alpha*m_db3_corr/(np.sqrt(v_db3_corr)+epsilon)


        if i%50==0:
            print("iteration",i)
            loss= categorical_cross_entropy_loss(A3,Y)
            accuracy=get_accuracy(get_predictions(A3),Y)
            print("Accuracy= ",accuracy, "  ","categorical_cross_entropy_loss= ",loss)
    return W1,b1,W2,b2,W3,b3
            

    
    

In [ ]:
W1,b1,W2,b2,W3,b3=gradient_with_adam_3(X_train,Y_train)

iteration 0
[5 5 5 ... 5 5 5] [3 8 5 ... 3 1 0]
Accuracy=  0.08819512195121951    categorical_cross_entropy_loss=  2.8890788397233957
iteration 50
[3 6 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.8600731707317073    categorical_cross_entropy_loss=  0.5582465586554555
iteration 100
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9106585365853659    categorical_cross_entropy_loss=  0.3442269442972689
iteration 150
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9303170731707318    categorical_cross_entropy_loss=  0.2655807773239318
iteration 200
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.9413658536585365    categorical_cross_entropy_loss=  0.22224451634081827
iteration 250
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.949    categorical_cross_entropy_loss=  0.19254917195001503
iteration 300
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.954780487804878    categorical_cross_entropy_loss=  0.16932716851824256
iteration 350
[3 8 5 ... 3 1 0] [3 8 5 ... 3 1 0]
Accuracy=  0.96041

In [ ]:
_,_,_,_,_,A3_test=forward_prop(W1,b1,W2,b2,W3,b3,X_test)
test_accuracy=get_accuracy(get_predictions(A3_test),Y_test)

print(f"Final Training Accuracy: 96.84%")
print(f"Final Test Accuracy:     {test_accuracy * 100:.2f}%")

[3 0 6 3 7 2 7 8 2 6 5 0 9 6 6 6 0 5 9 2 7 3 9 9 6 5 4 9 1 4 4 4 9 8 5 3 7
 3 8 1 8 9 9 6 8 4 4 9 2 8 9 1 0 2 4 7 0 6 8 6 3 5 5 6 6 4 0 6 6 7 1 6 1 4
 0 1 4 4 4 4 8 0 5 8 4 0 7 3 9 3 0 1 3 1 6 4 2 9 5 8 8 3 4 5 4 7 3 1 3 0 1
 7 4 0 2 1 7 9 9 2 3 0 3 1 2 7 2 8 5 1 2 7 8 7 3 7 0 3 0 3 5 5 0 0 6 7 9 0
 0 5 4 1 0 1 8 4 7 4 4 5 9 3 3 9 9 7 5 8 0 8 7 7 3 6 2 0 6 8 5 5 3 6 7 6 0
 3 7 5 8 4 6 6 8 2 2 3 7 8 3 3 9 8 8 4 9 4 2 1 9 5 6 5 8 6 0 4 8 2 7 6 2 7
 4 5 6 0 2 7 0 7 4 2 7 3 3 8 8 1 7 6 5 5 4 2 0 1 6 8 2 8 6 0 9 7 6 5 1 0 6
 7 3 2 1 6 5 3 4 3 8 6 7 6 3 4 1 6 2 4 8 3 1 3 1 5 1 1 2 4 9 1 1 4 6 0 6 1
 7 1 6 3 4 5 7 6 0 3 8 6 6 2 7 6 4 2 3 8 9 2 7 5 2 0 7 4 5 3 2 3 9 6 8 8 6
 5 0 8 2 7 7 2 5 6 2 6 0 5 3 3 0 2 9 1 7 0 3 3 5 9 9 4 7 9 7 1 0 9 9 0 8 4
 3 9 5 6 3 9 2 5 3 8 3 0 2 7 6 7 7 8 7 6 1 0 3 9 2 4 4 7 1 0 5 7 7 6 7 5 6
 3 7 6 3 4 0 3 4 0 5 8 0 0 9 4 3 4 4 4 6 7 0 4 0 2 0 5 6 1 8 2 9 3 5 1 0 2
 3 7 5 7 3 3 7 6 8 1 9 0 6 8 3 0 1 4 7 5 3 1 4 4 0 4 2 5 2 8 0 2 3 5 1 4 8
 6 6 3 5 3 6 2 0 3 4 0 9 

Ok so the things I Learned:
First I implemented the OG normal gradient descent optimization getting a training accuracy of 88.59 and test accuract of 88.61.

I later added the ADAM(Adaptive Moment Estimation) with beta1= 0.9 and beta2=0.99 and epsilon = 10^-10 which increased my accuracy to 90.6 and test accuracy to 91.4 ok slight jump but ya something is better than nothing. Also i added the Categorical Cross Entropy Loss for our ADAM to minimise it basically minimising the KL Divergence. 

Next I thought of taking it up 95+ maybe and i added a 3rd layer in the MLP with 784(input) - 128 - 64 - 10(output) and ladies and gentlemen it overfitted ok my bad. 

for ts I added L2 Regularization in the W3 matrix and it gave me a drop of 1.2 in training and test accuracies. BUT I WANT LOW. 
soooo.. I added L2 to W2 also but with reduced lambda of 0.005 earlier i used 0.01 in W3 and wallah my gap reduced to 0.6 with training accuracy of 96.84 and test accuracy of 96.20. 

Idk why I am doing this at 3 am but now am satisfied or maybe so frustrated that i gained enlightnment... nvm i should sleep okay. 
